# 07 - Analisis de estacionalidad mensual
Insumo: data/processed/suicidio_chihuahua_2019_2024.csv (output de
01_filter_chihuahua.ipynb, ya filtrado por Ent_resid == '08').

No se necesita denominador poblacional aqui: es una distribucion de
conteos por mes, no una tasa. Por eso se parte directo del output de
01, sin pasar por el merge de poblacion de 02.

OBJETIVO: probar si los casos de suicidio en Chihuahua (2019-2024) se
distribuyen uniformemente entre los 12 meses del anio, o si hay meses
con exceso, y comparar contra el patron reportado por Fernandez-Lopez
et al. (2021) para Chihuahua 2008-2018 (picos en junio y julio,
p < .002, asociado a temperatura).

NOTA METODOLOGICA: Fernandez-Lopez et al. usaron un modelo de series de
tiempo con temperatura ambiental como covariable. Aqui se usa una
prueba chi-cuadrado de bondad de ajuste (uniformidad mensual) -- un
primer acercamiento mas simple, no un modelo estacional equivalente.
Documentar esto como limitacion en el articulo si se usa este resultado.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
from cleaning_utils import test_estacionalidad_mensual, MESES_NOMBRE

df_chih = pd.read_csv(
    '../data/processed/suicidio_chihuahua_2019_2024.csv',
    encoding='utf-8', low_memory=False, dtype=str
)
print(f'Registros de Chihuahua cargados: {len(df_chih):,}')


## 1. Prueba de uniformidad mensual, agregado 2019-2024

In [ ]:
resultado = test_estacionalidad_mensual(df_chih, columna_mes='Mes_ocurr')

print('Calidad de datos (columna Mes_ocurr):')
for k, v in resultado['calidad_datos'].items():
    print(f'  {k}: {v:,}')
print()
print(f"chi2 = {resultado['chi2']}, p-value = {resultado['p_value']:.4g}")
print(f"Meses con mayor exceso sobre lo esperado: {resultado['meses_pico']}")
print()
resultado['tabla_mensual']


## 2. Estabilidad del patron por anio
Mismo enfoque que se uso para el RR de la Sierra Tarahumara (notebook
06): antes de confiar en el patron agregado 2019-2024, verificar que no
sea un artefacto de un solo anio atipico.

In [ ]:
resultados_por_anio = {}
for anio in sorted(df_chih['anio_dataset'].unique()):
    df_anio = df_chih[df_chih['anio_dataset'] == anio]
    r = test_estacionalidad_mensual(df_anio, columna_mes='Mes_ocurr')
    resultados_por_anio[anio] = r
    print(f"{anio}: chi2={r['chi2']}, p={r['p_value']:.4g}, "
          f"meses pico={r['meses_pico']}")


## 3. Comparacion contra Fernandez-Lopez et al. (2021)
Referencia: picos ciclicos en primavera-verano, mayores registros en
junio y julio, estacionalidad significativa en ambos sexos (p < .002),
periodo 2008-2018.

**Resultado agregado 2019-2024:** chi2 = 56.298, p = 4.48e-08
(n = 3,125 casos). Estacionalidad confirmada de forma solida: la
probabilidad de observar una distribucion mensual tan desigual bajo la
hipotesis de uniformidad es practicamente nula.

**Coincidencia parcial con Fernandez-Lopez et al. (2021):**
- Junio se mantiene como mes elevado en ambos estudios (+22.5% en
  nuestros datos).
- Julio, que en su estudio era el otro mes pico junto a junio, en
  2019-2024 esta practicamente en el nivel esperado (-3.2%) -- no se
  replica.
- Agosto (+24.0%) y septiembre (+11.7%) ganan protagonismo, sin ser
  mencionados como picos en el estudio de referencia.

**Interpretacion:** la estacionalidad como fenomeno persiste 11 anios
despues, pero el pico especifico parece haberse desplazado/ampliado de
junio-julio hacia junio-agosto-septiembre. Esto no contradice el
hallazgo original -- es una observacion nueva sobre como evoluciono el
patron, y vale la pena reportarla tal cual en vez de forzar la
coincidencia con el estudio previo.

**Limite de comparabilidad:** Fernandez-Lopez et al. usaron un modelo
de series de tiempo con temperatura ambiental como covariable; aqui se
uso una prueba de uniformidad mas simple (ver nota metodologica en la
introduccion del notebook). La coincidencia/divergencia de meses pico
es comparable a nivel descriptivo, pero no es una replica metodologica
exacta de su analisis.

## 4. Guardar tabla mensual

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
resultado['tabla_mensual'].to_csv(
    '../data/processed/estacionalidad_mensual_chihuahua_2019_2024.csv',
    index=False, encoding='utf-8'
)
print('Guardado: estacionalidad_mensual_chihuahua_2019_2024.csv')


## 5. Hallazgos

**Resultado principal:** hay estacionalidad significativa en los
suicidios de Chihuahua 2019-2024 (chi2 = 56.298, p = 4.48e-08,
n = 3,125). Los meses con mayor exceso sobre lo esperado son agosto
(+24.0%), junio (+22.5%) y septiembre (+11.7%); los de mayor deficit
son noviembre (-20.5%) y enero (-14.8%).

**Estabilidad anio con anio:** la prueba es significativa (alfa = .05)
en 4 de 6 anios (2019, 2020, 2021, 2023); no alcanza significancia en
2022 (p = .334) ni 2024 (p = .123). Esto NO se interpreta como ausencia
de estacionalidad en esos dos anios: cada anio tiene ~500 casos
repartidos en 12 meses (~40-45 por mes), una muestra bastante mas chica
que el agregado, por lo que la prueba pierde poder estadistico. Un
efecto real del mismo tamano puede no alcanzar significancia solo por
tener menos casos -- no equivale a que el patron haya desaparecido.
Los meses junio y agosto aparecen en el top-3 de meses pico en 4 de los
6 anios cada uno, incluyendo anios donde la prueba global no fue
significativa -- es la evidencia mas solida de consistencia disponible
con este diseno.

**Comparacion con Fernandez-Lopez et al. (2021):** la estacionalidad
como fenomeno se confirma 11 anios despues, pero el pico especifico se
desplazo de junio-julio hacia junio-agosto-septiembre; julio deja de
ser mes elevado en este periodo (ver seccion 3 para el detalle).

**Aclaracion metodologica:** la columna `meses_pico` es descriptiva
(ranking por exceso_pct), no una prueba estadistica independiente por
mes. La unica prueba formal es el chi-cuadrado global de uniformidad;
no se le debe atribuir un p-value implicito a un mes individual del
ranking.

**Calidad de datos:** registros con mes no especificado o nulo
excluidos del analisis segun columna 'calidad_datos' de la celda de
resultados -- documentar la proporcion exacta en methodology.md si no
es trivial.

**Pendiente para explorar despues (no bloqueante para este notebook):**
separar 2019-2021 vs. 2022-2024 para ver si el desplazamiento de julio
hacia agosto-septiembre es gradual o abrupto; y, como extension futura,
incorporar temperatura mensual (CONAGUA/SMN) como covariable en un
modelo Poisson equivalente al de Fernandez-Lopez et al.